In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
BASE_URL = "https://books.toscrape.com/"

In [3]:
categories = [
    "Thriller",
    "Mystery",
    "Historical Fiction"
]

In [4]:
# Cell 4: Functions to scrape books from selected categories

from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"


def get_soup(url):
    """
    Sends a GET request and returns a BeautifulSoup object.
    """
    response = requests.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")


def get_category_links():
    """
    Returns a dictionary:
    {
        "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
        ...
    }
    """
    soup = get_soup(BASE_URL)

    category_links = {}

    categories = soup.select("div.side_categories ul li ul li a")

    for category in categories:
        name = category.text.strip()
        href = category["href"]

        category_links[name] = urljoin(BASE_URL, href)

    return category_links


def scrape_category(category_name, category_url):
    """
    Scrapes every page in one category.
    Returns a list of dictionaries.
    """

    books = []

    while True:

        soup = get_soup(category_url)

        book_cards = soup.select("article.product_pod")

        for book in book_cards:

            # Title
            title = book.h3.a["title"]

            # Price (GBP)
            price = book.select_one("p.price_color").text.strip()

            # Rating (text)
            rating = book.select_one("p.star-rating")["class"][1]

            # Book page link
            book_link = urljoin(
                category_url,
                book.h3.a["href"]
            )

            # Visit individual book page
            book_soup = get_soup(book_link)

            # Availability
            availability = (
                book_soup
                .select_one("p.instock.availability")
                .text
                .strip()
            )

            books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category_name
            })

        # Next page
        next_button = soup.select_one("li.next a")

        if next_button:
            category_url = urljoin(category_url, next_button["href"])
        else:
            break

    return books

In [5]:
# Cell 5: Scrape selected categories

# Categories to scrape
selected_categories = [
    "Thriller",
    "Mystery",
    "Historical Fiction"
]

# Get all category URLs
category_links = get_category_links()

# Store all books here
all_books = []

# Scrape each selected category
for category in selected_categories:
    
    print(f"Scraping category: {category}")
    
    books = scrape_category(category, category_links[category])
    
    print(f"Collected {len(books)} books\n")
    
    all_books.extend(books)

# Convert to DataFrame
raw_df = pd.DataFrame(all_books)

print("-" * 50)
print(f"Total books scraped: {len(raw_df)}")
print("-" * 50)

# Display first few rows
raw_df.head()

Scraping category: Thriller
Collected 11 books

Scraping category: Mystery
Collected 32 books

Scraping category: Historical Fiction
Collected 26 books

--------------------------------------------------
Total books scraped: 69
--------------------------------------------------


,title,price,star_rating,availability,category
0,In Her Wake,Â£12.84,One,In stock (19 available),Thriller
1,The Elephant Tree,Â£23.82,Five,In stock (18 available),Thriller
2,Behind Closed Doors,Â£52.22,Four,In stock (18 available),Thriller
3,You (You #1),Â£43.61,Five,In stock (14 available),Thriller
4,The Guilty (Will Robie #4),Â£13.82,Two,In stock (14 available),Thriller


In [6]:
# Cell 6: Verify scraped data

print("Dataset Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

print("\nBooks per Category:")
print(raw_df["category"].value_counts())

print("\nSample Data:")
display(raw_df.head())

Dataset Shape: (69, 5)

Columns:
['title', 'price', 'star_rating', 'availability', 'category']

Books per Category:
category
Mystery               32
Historical Fiction    26
Thriller              11
Name: count, dtype: int64

Sample Data:


,title,price,star_rating,availability,category
0,In Her Wake,Â£12.84,One,In stock (19 available),Thriller
1,The Elephant Tree,Â£23.82,Five,In stock (18 available),Thriller
2,Behind Closed Doors,Â£52.22,Four,In stock (18 available),Thriller
3,You (You #1),Â£43.61,Five,In stock (14 available),Thriller
4,The Guilty (Will Robie #4),Â£13.82,Two,In stock (14 available),Thriller


In [7]:
print(f"Dataset saved successfully!")
print(f"Location: data/raw/raw_books.csv")
print(f"Total Books: {len(raw_df)}")

Dataset saved successfully!
Location: data/raw/raw_books.csv
Total Books: 69


In [8]:
# Cell 8: Data Cleaning

import numpy as np

clean_df = raw_df.copy()

# -------------------------------
# Convert price to float (GBP)
# -------------------------------

clean_df["price_gbp"] = (
    clean_df["price"]
    .str.replace("£", "", regex=False)
)

clean_df["price_gbp"] = pd.to_numeric(
    clean_df["price_gbp"],
    errors="coerce"
)

# -------------------------------
# Convert rating text to integer
# -------------------------------

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

clean_df["rating"] = clean_df["star_rating"].map(rating_map)

# -------------------------------
# Convert availability to boolean
# -------------------------------

clean_df["in_stock"] = (
    clean_df["availability"]
    .str.contains("In stock", case=False, na=False)
)

# -------------------------------
# Median imputation
# -------------------------------

clean_df["price_gbp"] = clean_df["price_gbp"].fillna(
    clean_df["price_gbp"].median()
)

clean_df["rating"] = clean_df["rating"].fillna(
    clean_df["rating"].median()
)

clean_df["rating"] = clean_df["rating"].astype(int)

print("Cleaning completed successfully!")

Cleaning completed successfully!


In [9]:
# Cell 9: Verify cleaned data

print(clean_df.dtypes)

print("\n")

display(clean_df.head())

title               str
price               str
star_rating         str
availability        str
category            str
price_gbp       float64
rating            int64
in_stock           bool
dtype: object




,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,In Her Wake,Â£12.84,One,In stock (19 available),Thriller,NaN,1,True
1,The Elephant Tree,Â£23.82,Five,In stock (18 available),Thriller,NaN,5,True
2,Behind Closed Doors,Â£52.22,Four,In stock (18 available),Thriller,NaN,4,True
3,You (You #1),Â£43.61,Five,In stock (14 available),Thriller,NaN,5,True
4,The Guilty (Will Robie #4),Â£13.82,Two,In stock (14 available),Thriller,NaN,2,True


In [36]:
# Cell 10: Save cleaned dataset

clean_df.to_csv(
    "data_pipeline/data/cleaned/clean_books.csv",
    index=False
)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!


In [12]:
# Cell 11: Convert GBP to INR

# Fixed project-defined exchange rate
GBP_TO_INR = 105.50

# Create new column
clean_df["price_inr"] = (clean_df["price_gbp"] * GBP_TO_INR).round(2)

print("GBP to INR conversion completed successfully!")

# Display sample
display(clean_df[["title", "price_gbp", "price_inr"]].head())

GBP to INR conversion completed successfully!


,title,price_gbp,price_inr
0,In Her Wake,NaN,NaN
1,The Elephant Tree,NaN,NaN
2,Behind Closed Doors,NaN,NaN
3,You (You #1),NaN,NaN
4,The Guilty (Will Robie #4),NaN,NaN


In [13]:
print(clean_df.dtypes)

title               str
price               str
star_rating         str
availability        str
category            str
price_gbp       float64
rating            int64
in_stock           bool
price_inr       float64
dtype: object


In [38]:
clean_df.to_csv(
    "data_pipeline/data/cleaned/clean_books.csv",
    index=False
)

print("Updated cleaned dataset saved successfully!")

Updated cleaned dataset saved successfully!


In [39]:
import sqlite3

db_path = "data_pipeline/database/books.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Database connected successfully!")

Database connected successfully!


In [40]:
# Cell 15: Create Database Tables

# Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON;")

# -------------------------
# Categories Table
# -------------------------

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")

# -------------------------
# Books Table
# -------------------------

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Tables created successfully!")

Tables created successfully!


In [41]:
# Cell 16: Verify Tables

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print("Tables in database:")

for table in tables:
    print(table[0])

Tables in database:
categories
sqlite_sequence
books


In [42]:
# Cell 17: Insert Categories

categories = clean_df["category"].unique()

for category in categories:
    cursor.execute("""
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
    """, (category,))

conn.commit()

print("Categories inserted successfully!")

Categories inserted successfully!


In [43]:
# Cell 18: Verify Categories

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

display(categories_df)

,category_id,category_name
0,1,Thriller
1,2,Mystery
2,3,Historical Fiction


In [44]:
# Cell 19: Insert Books

for _, row in clean_df.iterrows():

    # Find the category_id
    cursor.execute("""
        SELECT category_id
        FROM categories
        WHERE category_name = ?
    """, (row["category"],))

    category_id = cursor.fetchone()[0]

    # Insert the book
    cursor.execute("""
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["in_stock"]),   # SQLite stores booleans as 0/1
        category_id
    ))

conn.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [45]:
# Cell 20: Verify Books

books_df = pd.read_sql(
    "SELECT * FROM books LIMIT 10",
    conn
)

display(books_df)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,In Her Wake,None,None,1,1,1
1,2,The Elephant Tree,None,None,5,1,1
2,3,Behind Closed Doors,None,None,4,1,1
3,4,You (You #1),None,None,5,1,1
4,5,The Guilty (Will Robie #4),None,None,2,1,1
5,6,The 14th Colony (Cotton Malone #11),None,None,1,1,1
6,7,Give It Back,None,None,2,1,1
7,8,Killing Floor (Jack Reacher #1),None,None,4,1,1
8,9,The Bone Hunters (Lexy Vaughan & Steven Macaul...,None,None,3,1,1
9,10,Far From True (Promise Falls Trilogy #2),None,None,2,1,1


In [46]:
# Cell 21: Verify Database

cursor.execute("SELECT COUNT(*) FROM categories")
category_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM books")
book_count = cursor.fetchone()[0]

print("=" * 40)
print("DATABASE VERIFICATION")
print("=" * 40)

print(f"Categories : {category_count}")
print(f"Books      : {book_count}")

print("\nDatabase created successfully!")

DATABASE VERIFICATION
Categories : 3
Books      : 138

Database created successfully!


In [47]:
# Query 1: Books with rating greater than 3

query1 = """
SELECT title, rating
FROM books
WHERE rating > 3;
"""

print("Query 1:")
print(query1)

result1 = pd.read_sql(query1, conn)
display(result1)

Query 1:

SELECT title, rating
FROM books
WHERE rating > 3;



,title,rating
0,The Elephant Tree,5
1,Behind Closed Doors,4
2,You (You #1),5
3,Killing Floor (Jack Reacher #1),4
4,Sharp Objects,4
5,The Past Never Ends,4
6,The Murder of Roger Ackroyd (Hercule Poirot #4),4
7,A Time of Torment (Charlie Parker #14),5
8,Murder at the 42nd Street Library (Raymond Amb...,4
9,What Happened on Beale Street (Secrets of the ...,5


In [48]:
# Query 2: Top 10 most expensive books

query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

print("Query 2:")
print(query2)

result2 = pd.read_sql(query2, conn)
display(result2)

Query 2:

SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10;



,title,price_gbp
0,In Her Wake,None
1,The Elephant Tree,None
2,Behind Closed Doors,None
3,You (You #1),None
4,The Guilty (Will Robie #4),None
5,The 14th Colony (Cotton Malone #11),None
6,Give It Back,None
7,Killing Floor (Jack Reacher #1),None
8,The Bone Hunters (Lexy Vaughan & Steven Macaul...,None
9,Far From True (Promise Falls Trilogy #2),None


In [49]:
# Query 3: Distinct Categories

query3 = """
SELECT DISTINCT category_name
FROM categories;
"""

print("Query 3:")
print(query3)

result3 = pd.read_sql(query3, conn)
display(result3)

Query 3:

SELECT DISTINCT category_name
FROM categories;



,category_name
0,Historical Fiction
1,Mystery
2,Thriller


In [50]:
# Query 4: Books priced between £20 and £40

query4 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
"""

print("Query 4:")
print(query4)

result4 = pd.read_sql(query4, conn)
display(result4)

Query 4:

SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40;



,title,price_gbp


In [51]:
# Query 5: Join Books and Categories

query5 = """
SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id;
"""

print("Query 5:")
print(query5)

join_sql = pd.read_sql(query5, conn)
display(join_sql)

Query 5:

SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id;



,title,category_name,rating,price_gbp
0,In Her Wake,Thriller,1,None
1,The Elephant Tree,Thriller,5,None
2,Behind Closed Doors,Thriller,4,None
3,You (You #1),Thriller,5,None
4,The Guilty (Will Robie #4),Thriller,2,None
...,...,...,...,...
133,While You Were Mine,Historical Fiction,5,None
134,The Secret Healer,Historical Fiction,3,None
135,Starlark,Historical Fiction,3,None
136,Lost Among the Living,Historical Fiction,4,None


In [52]:
# Query 6: Top 10 highest-rated books by category

query6 = """
SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC,
         b.price_gbp DESC
LIMIT 10;
"""

print("Query 6:")
print(query6)

result6 = pd.read_sql(query6, conn)
display(result6)

Query 6:

SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC,
         b.price_gbp DESC
LIMIT 10;



,title,category_name,rating,price_gbp
0,The Elephant Tree,Thriller,5,None
1,You (You #1),Thriller,5,None
2,A Time of Torment (Charlie Parker #14),Mystery,5,None
3,What Happened on Beale Street (Secrets of the ...,Mystery,5,None
4,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,None
5,The Silkworm (Cormoran Strike #2),Mystery,5,None
6,The Girl You Lost,Mystery,5,None
7,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,None
8,Mrs. Houdini,Historical Fiction,5,None
9,The Passion of Dolssa,Historical Fiction,5,None


In [53]:
# Query 7: Average Book Price by Category

query7 = """
SELECT
    c.category_name,
    COUNT(b.book_id) AS total_books,
    ROUND(AVG(b.price_gbp), 2) AS average_price_gbp,
    ROUND(AVG(b.price_inr), 2) AS average_price_inr
FROM books b
JOIN categories c
ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY average_price_gbp DESC;
"""

print("Query 7:")
print(query7)

result7 = pd.read_sql(query7, conn)
display(result7)

Query 7:

SELECT
    c.category_name,
    COUNT(b.book_id) AS total_books,
    ROUND(AVG(b.price_gbp), 2) AS average_price_gbp,
    ROUND(AVG(b.price_inr), 2) AS average_price_inr
FROM books b
JOIN categories c
ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY average_price_gbp DESC;



,category_name,total_books,average_price_gbp,average_price_inr
0,Thriller,22,None,None
1,Mystery,64,None,None
2,Historical Fiction,52,None,None


In [54]:
# Read Query 2 into a DataFrame

query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

top_expensive_df = pd.read_sql(query2, conn)

print("Top 10 Most Expensive Books")
display(top_expensive_df)

Top 10 Most Expensive Books


,title,price_gbp
0,In Her Wake,None
1,The Elephant Tree,None
2,Behind Closed Doors,None
3,You (You #1),None
4,The Guilty (Will Robie #4),None
5,The 14th Colony (Cotton Malone #11),None
6,Give It Back,None
7,Killing Floor (Jack Reacher #1),None
8,The Bone Hunters (Lexy Vaughan & Steven Macaul...,None
9,Far From True (Promise Falls Trilogy #2),None


In [55]:
# Read Query 7 into a DataFrame

query7 = """
SELECT
    c.category_name,
    COUNT(b.book_id) AS total_books,
    ROUND(AVG(b.price_gbp),2) AS average_price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id
GROUP BY c.category_name;
"""

category_summary_df = pd.read_sql(query7, conn)

print("Category Summary")
display(category_summary_df)

Category Summary


,category_name,total_books,average_price_gbp
0,Historical Fiction,52,None
1,Mystery,64,None
2,Thriller,22,None


In [56]:
books_df = pd.read_sql("SELECT * FROM books", conn)

categories_df = pd.read_sql("SELECT * FROM categories", conn)

In [57]:
merge_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

merge_df = merge_df[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp"
    ]
]

print("JOIN using pd.merge()")
display(merge_df)

JOIN using pd.merge()


,title,category_name,rating,price_gbp
0,In Her Wake,Thriller,1,None
1,The Elephant Tree,Thriller,5,None
2,Behind Closed Doors,Thriller,4,None
3,You (You #1),Thriller,5,None
4,The Guilty (Will Robie #4),Thriller,2,None
...,...,...,...,...
133,While You Were Mine,Historical Fiction,5,None
134,The Secret Healer,Historical Fiction,3,None
135,Starlark,Historical Fiction,3,None
136,Lost Among the Living,Historical Fiction,4,None


In [58]:
join_query = """
SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
JOIN categories c
ON b.category_id = c.category_id;
"""

sql_join_df = pd.read_sql(join_query, conn)

print("JOIN using SQL")
display(sql_join_df)

JOIN using SQL


,title,category_name,rating,price_gbp
0,In Her Wake,Thriller,1,None
1,The Elephant Tree,Thriller,5,None
2,Behind Closed Doors,Thriller,4,None
3,You (You #1),Thriller,5,None
4,The Guilty (Will Robie #4),Thriller,2,None
...,...,...,...,...
133,While You Were Mine,Historical Fiction,5,None
134,The Secret Healer,Historical Fiction,3,None
135,Starlark,Historical Fiction,3,None
136,Lost Among the Living,Historical Fiction,4,None


In [59]:
# Sort both DataFrames for a fair comparison

sql_sorted = sql_join_df.sort_values(
    by=["title", "category_name"]
).reset_index(drop=True)

merge_sorted = merge_df.sort_values(
    by=["title", "category_name"]
).reset_index(drop=True)

print("Are both outputs identical?")

print(sql_sorted.equals(merge_sorted))

Are both outputs identical?
True
